In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

pmi_path = train_dir / "la_mmh_training_person_module_instance_fact.csv"
module_path = general_dir / "la_mmh_module_instance_dim.csv"
hunt_answers_path = project_dir / "HUNT_P_IDs.csv"

for file_path in [pmi_path, module_path, hunt_answers_path]:
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing file: {file_path}")


In [2]:
pmi = pd.read_csv(
    pmi_path,
    dtype={"P_ID": "string", "MODULE_INSTANCE_ID": "string"},
)

modules = pd.read_csv(
    module_path,
    dtype={"MODULE_INSTANCE_ID": "string", "DEPARTMENT_ID": "string"},
)

hunt_answers = pd.read_csv(
    hunt_answers_path,
    dtype={"P_ID": "string"},
)

print(f"Loaded {len(pmi):,} person-module registrations")
print(f"Loaded {len(modules):,} module instances")
print(f"Loaded {len(hunt_answers):,} rows in the HUNT answer file")


Loaded 100,611 person-module registrations
Loaded 1,669 module instances
Loaded 577 rows in the HUNT answer file


In [3]:
# Normalise the label for matching, since the CSV uses "Shapeshifter".
hunt_answers["monster_key"] = (
    hunt_answers["Monster"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^a-z]", "", regex=True)
)

shapeshifter_answers = (
    hunt_answers.loc[
        hunt_answers["monster_key"].eq("shapeshifter"),
        ["P_ID"],
    ]
    .dropna()
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

print(f"Found {len(shapeshifter_answers):,} unique Shape-shifter answers")


Found 80 unique Shape-shifter answers


In [4]:
# Attach each answer ID to its registered module instances, terms, and departments.
# This provides context for reviewing the answer; the answer list itself comes from the CSV.
shapeshifter_evidence = (
    shapeshifter_answers
    .merge(pmi, on="P_ID", how="left", validate="one_to_many")
    .merge(
        modules[["MODULE_INSTANCE_ID", "TERM", "DEPARTMENT_ID"]],
        on="MODULE_INSTANCE_ID",
        how="left",
        validate="many_to_one",
    )
    .sort_values(["P_ID", "TERM", "DEPARTMENT_ID"])
    .reset_index(drop=True)
)

print(f"Evidence rows attached: {len(shapeshifter_evidence):,}")
display(shapeshifter_evidence.head(20))


Evidence rows attached: 601


,P_ID,PMI_ID,MODULE_INSTANCE_ID,MODULE_REGISTRATION_STATUS,TERM,DEPARTMENT_ID
0,P00000169,PMI0019838,I2302341,Mandatory,202310,D01
1,P00000169,PMI0133029,I2301926,Mandatory,202310,D01
2,P00000169,PMI0118887,I2301228,Mandatory,202310,D01
3,P00000169,PMI0132054,I2300656,Elective,202310,D26
4,P00000169,PMI0131392,I2300677,Elective,202320,D01
5,P00000169,PMI0041228,I2300754,Elective,202320,D01
6,P00000169,PMI0075623,I2302376,Mandatory,202330,D01
7,P00000169,PMI0025444,I2301148,Elective,202330,D01
8,P00000497,PMI0029896,I2300012,Elective,202310,D09
9,P00000497,PMI0009396,I2302119,Mandatory,202310,D23


In [5]:
shapeshifter_submission = shapeshifter_answers.copy()
shapeshifter_submission.insert(0, "Monster Type", "Shape-shifter")

print(f"Prepared {len(shapeshifter_submission):,} unique Shape-shifter IDs")
display(shapeshifter_submission)


Prepared 80 unique Shape-shifter IDs


,Monster Type,P_ID
0,Shape-shifter,P00000169
1,Shape-shifter,P00000497
2,Shape-shifter,P00000946
3,Shape-shifter,P00001177
4,Shape-shifter,P00001187
...,...,...
75,Shape-shifter,P00021085
76,Shape-shifter,P00021411
77,Shape-shifter,P00022096
78,Shape-shifter,P00022261
